# 02 — Data Collection
Goal: pull all available match and player data from the API, flatten it, and save as parquet files under `data/processed/`.

In [1]:
import sys
sys.path.insert(0, '..')

import datetime
import pandas as pd
from pathlib import Path
from src.api.client import PadelAPIClient
from src.processing.flatten import flatten_match, flatten_player, flatten_tournament

client = PadelAPIClient(requests_per_minute=10)
PROCESSED_DIR = Path('../data/processed')
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

today = datetime.date.today().isoformat()
print(f'Collecting data up to {today}')

## 1. Players
Fetch all players across all pages.

In [2]:
raw_players = client.get_all_players()
print(f'Fetched {len(raw_players)} players')

df_players = pd.DataFrame([flatten_player(p) for p in raw_players])
df_players['birthdate'] = pd.to_datetime(df_players['birthdate'], errors='coerce')
print(df_players.shape)
df_players.head()

  page 1: +15 items (total so far: 15/909)
  page 2: +15 items (total so far: 30/909)
  page 3: +15 items (total so far: 45/909)
  page 4: +15 items (total so far: 60/909)
  page 5: +15 items (total so far: 75/909)
  429 rate limited — waiting 30s before retry (3 left)...
  page 6: +15 items (total so far: 90/909)
  page 7: +15 items (total so far: 105/909)
  page 8: +15 items (total so far: 120/909)
  page 9: +15 items (total so far: 135/909)
  page 10: +15 items (total so far: 150/909)
  429 rate limited — waiting 30s before retry (3 left)...
  page 11: +15 items (total so far: 165/909)
  page 12: +15 items (total so far: 180/909)
  page 13: +15 items (total so far: 195/909)
  page 14: +15 items (total so far: 210/909)
  page 15: +15 items (total so far: 225/909)
  429 rate limited — waiting 30s before retry (3 left)...
  page 16: +15 items (total so far: 240/909)
  page 17: +15 items (total so far: 255/909)
  page 18: +15 items (total so far: 270/909)
  page 19: +15 items (total so 

,player_id,name,short_name,category,ranking,points,nationality,birthdate,age,height,hand,side
0,371,Aaron Segura,Segura,men,799.0,22.0,US,2004-12-03,21.0,NaN,None,None
1,339,Abdulaziz Redha,Redha,men,895.0,18.0,KW,1998-02-25,28.0,NaN,None,None
2,344,Abdulaziz Saadon Alkuwari,Saadon,men,2734.0,NaN,QA,1979-11-13,46.0,179.0,None,drive
3,186,Abdulla Alhijji,Alhijji,men,525.0,49.0,QA,1990-01-25,36.0,177.0,None,backhand
4,336,Abdullah Alabdullah,Alabdullah,men,813.0,22.0,SA,2002-09-04,23.0,NaN,None,None


## 2. Matches
Fetch all completed matches (paginated). The free tier covers the last 6 months.

In [5]:
raw_matches = client.get_all_matches(before_date=today)
print(f'Fetched {len(raw_matches)} matches total')

finished = [m for m in raw_matches if m.get('status') == 'finished']
print(f'Finished matches: {len(finished)}')

df_matches = pd.DataFrame([flatten_match(m) for m in finished])
df_matches['played_at'] = pd.to_datetime(df_matches['played_at'], errors='coerce')
print(df_matches.shape)
df_matches.head()

  page 1: +15 items (total so far: 15/6997)
  page 2: +15 items (total so far: 30/6997)
  page 3: +15 items (total so far: 45/6997)
  page 4: +15 items (total so far: 60/6997)
  page 5: +15 items (total so far: 75/6997)
  429 rate limited — waiting 30s before retry (3 left)...
  page 6: +15 items (total so far: 90/6997)
  page 7: +15 items (total so far: 105/6997)
  page 8: +15 items (total so far: 120/6997)
  page 9: +15 items (total so far: 135/6997)
  page 10: +15 items (total so far: 150/6997)
  429 rate limited — waiting 30s before retry (3 left)...
  page 11: +15 items (total so far: 165/6997)
  page 12: +15 items (total so far: 180/6997)
  page 13: +15 items (total so far: 195/6997)
  page 14: +15 items (total so far: 210/6997)
  page 15: +15 items (total so far: 225/6997)
  429 rate limited — waiting 30s before retry (3 left)...
  page 16: +15 items (total so far: 240/6997)
  page 17: +15 items (total so far: 255/6997)
  page 18: +15 items (total so far: 270/6997)
  page 19: +1

AttributeError: 'str' object has no attribute 'get'

In [7]:
import importlib
import src.processing.flatten as flatten_mod
importlib.reload(flatten_mod)
from src.processing.flatten import flatten_match, flatten_player, flatten_tournament


In [8]:
finished = [m for m in raw_matches if m.get('status') == 'finished']
print(f'Finished matches: {len(finished)}')

df_matches = pd.DataFrame([flatten_match(m) for m in finished])
df_matches['played_at'] = pd.to_datetime(df_matches['played_at'], errors='coerce')
print(df_matches.shape)
df_matches.head()


Finished matches: 5170
(5170, 16)


,match_id,tournament_id,category,round,round_name,played_at,status,winner,duration,t1_p1,t1_p2,t2_p1,t2_p2,sets_won_t1,sets_won_t2,n_sets
0,7417,728,men,32,Round of 64,2026-03-24,finished,team_1,01:12,148.0,64.0,107.0,159.0,2,0,2
1,7422,728,men,32,Round of 64,2026-03-24,finished,team_1,02:07,452.0,98.0,163.0,465.0,2,0,2
2,7429,728,men,32,Round of 64,2026-03-24,finished,team_2,01:17,197.0,250.0,151.0,199.0,0,2,2
3,7430,728,men,32,Round of 64,2026-03-24,finished,team_2,01:58,284.0,228.0,92.0,211.0,1,2,3
4,7433,728,men,32,Round of 64,2026-03-24,finished,team_2,01:04,1248.0,1249.0,157.0,210.0,0,2,2


## 3. Tournaments
Fetch metadata for every tournament referenced in the match data.

In [9]:
tournament_ids = df_matches['tournament_id'].dropna().astype(int).unique()
print(f'Unique tournaments in match data: {len(tournament_ids)}')

raw_tournaments = [client.get_tournament(int(tid)) for tid in tournament_ids]
df_tournaments = pd.DataFrame([flatten_tournament(t) for t in raw_tournaments])
df_tournaments['start_date'] = pd.to_datetime(df_tournaments['start_date'], errors='coerce')
df_tournaments['end_date'] = pd.to_datetime(df_tournaments['end_date'], errors='coerce')
print(df_tournaments.shape)
df_tournaments.head()

Unique tournaments in match data: 95
  429 rate limited — waiting 30s before retry (3 left)...
  429 rate limited — waiting 30s before retry (3 left)...
  429 rate limited — waiting 30s before retry (3 left)...


HTTPError: 404 Client Error: Not Found for url: https://padelapi.org/api/tournaments/511

In [11]:
raw_tournaments = []
for tid in tournament_ids:
    try:
        raw_tournaments.append(client.get_tournament(int(tid)))
    except Exception as e:
        print(f"  Skipping tournament {tid}: {e}")

df_tournaments = pd.DataFrame([flatten_tournament(t) for t in raw_tournaments])
df_tournaments['start_date'] = pd.to_datetime(df_tournaments['start_date'], errors='coerce')
df_tournaments['end_date'] = pd.to_datetime(df_tournaments['end_date'], errors='coerce')
print(df_tournaments.shape)
df_tournaments.head()


  Skipping tournament 511: 404 Client Error: Not Found for url: https://padelapi.org/api/tournaments/511
  429 rate limited — waiting 30s before retry (3 left)...
  429 rate limited — waiting 30s before retry (3 left)...
  429 rate limited — waiting 30s before retry (3 left)...
  429 rate limited — waiting 30s before retry (3 left)...
  429 rate limited — waiting 30s before retry (3 left)...
  429 rate limited — waiting 30s before retry (3 left)...
  429 rate limited — waiting 30s before retry (3 left)...
  429 rate limited — waiting 30s before retry (3 left)...
  429 rate limited — waiting 30s before retry (3 left)...
  429 rate limited — waiting 30s before retry (3 left)...
  429 rate limited — waiting 30s before retry (3 left)...
  429 rate limited — waiting 30s before retry (3 left)...
  429 rate limited — waiting 30s before retry (3 left)...
  429 rate limited — waiting 30s before retry (3 left)...
  429 rate limited — waiting 30s before retry (3 left)...
(94, 9)


,tournament_id,season_id,name,location,country,level,status,start_date,end_date
0,728,5,Miami P1 2026,Miami,US,p1,live,2026-03-23,2026-03-29
1,727,5,Cancún P2 2026,Cancún,MX,p2,finished,2026-03-16,2026-03-22
2,726,5,Gijón P2 2026,Gijón,ES,p2,finished,2026-03-02,2026-03-08
3,778,6,FIP Gold Ponta Delgada,Açores,PT,fip_gold,finished,2026-03-04,2026-03-08
4,725,5,Riyadh Season P1 2026,Riyadh,SA,p1,finished,2026-02-07,2026-02-14


## 4. Save to Parquet

In [12]:
df_players.to_parquet(PROCESSED_DIR / 'players.parquet', index=False)
df_matches.to_parquet(PROCESSED_DIR / 'matches.parquet', index=False)
df_tournaments.to_parquet(PROCESSED_DIR / 'tournaments.parquet', index=False)
print('Saved:')
print(f'  players.parquet    {len(df_players):>5} rows')
print(f'  matches.parquet    {len(df_matches):>5} rows')
print(f'  tournaments.parquet {len(df_tournaments):>4} rows')

Saved:
  players.parquet      909 rows
  matches.parquet     5170 rows
  tournaments.parquet   94 rows


## 5. Dataset Summary

In [13]:
# Date range
print('Match date range:', df_matches['played_at'].min().date(), '→', df_matches['played_at'].max().date())
print()

# Matches by category
print('Matches by category:')
print(df_matches['category'].value_counts().to_string())
print()

# Matches by round
print('Matches by round:')
print(df_matches[['round', 'round_name']].value_counts().sort_index().to_string())
print()

# Null rates for key columns
key_cols = ['winner', 'sets_won_t1', 'sets_won_t2', 't1_p1', 't1_p2', 't2_p1', 't2_p2']
null_pct = df_matches[key_cols].isna().mean().mul(100).round(1)
print('Null % for key columns:')
print(null_pct.to_string())

Match date range: 2023-02-22 → 2026-03-24

Matches by category:
category
men      3286
women    1884

Matches by round:
round  round_name 
1      Finals          152
2      Semifinals      308
4      Quarter         618
8      Round of 16    1189
16     Round of 32    2124
32     Round of 64     779

Null % for key columns:
winner         0.0
sets_won_t1    0.0
sets_won_t2    0.0
t1_p1          1.1
t1_p2          1.1
t2_p1          1.4
t2_p2          1.4


In [4]:
df_players.to_parquet('../data/processed/players.parquet', index=False)
print(f'Saved {len(df_players)} players')


Saved 909 players
